# 01 · Atributos de Polyvore desde el texto

Paso 3 de la etapa 3. A cada prenda de los outfits vestibles (paso 2) se le sacan del **nombre en inglés** los atributos del esquema común: tono, familia de color, patrón, material, estilo, calce, época y largo. Las reglas y los diccionarios están en `src/modelo_fly/atributos.py`.

**Entradas:** `data/processed/hito3/polyvore_items.csv` y `polyvore_outfits.csv` (de `00_polyvore_descarga`).
**Salida:** `data/processed/hito3/polyvore_items_atributos.csv` (no se sube a git).

**Supuestos (declararlos en el informe):**
- Si el nombre no menciona un patrón, la prenda es **lisa**; si no menciona una época, es **contemporánea**.
- Si el nombre no dice estilo, calce, material o largo, se usa la **pista de la categoría** (*Skinny Jeans* → ajustado, *Gowns* → formal, *Jeans* → denim, *Mini Skirts* → corto).
- Los **metales** (dorado, plateado) son neutros en joyería y accesorios, y acento fuerte en ropa.
- Calce y largo solo aplican a la ropa (arriba, abajo, vestido, exterior).

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("No se encontró data/raw. Abre el notebook dentro del repositorio.")
sys.path.insert(0, str(ROOT / "src"))

from modelo_fly.atributos import SLOTS_ROPA, extraer
from modelo_fly.esquema import valores_validos

PROC = ROOT / "data" / "processed" / "hito3"
SALIDA = PROC / "polyvore_items_atributos.csv"
ATRIBUTOS = ["tono", "familia_color", "patron", "material", "estilo", "calce", "epoca", "largo"]

items = pd.read_csv(PROC / "polyvore_items.csv", dtype=str, keep_default_na=False)
outfits = pd.read_csv(PROC / "polyvore_outfits.csv", dtype=str, keep_default_na=False)
validos = set(outfits.loc[outfits["valido"] == "True", "set_id"])
items = items[items["set_id"].isin(validos) & (items["slot"] != "excluido")].reset_index(drop=True)
print("Ítems de moda en outfits vestibles:", len(items))

Ítems de moda en outfits vestibles: 81372


## 1. Extracción

In [2]:
attrs = pd.DataFrame([
    extraer(n, c, s, l) for n, c, s, l in zip(items["name"], items["categoria"], items["slot"], items["largo_categoria"])
])
items = pd.concat([items, attrs], axis=1)

for a in ATRIBUTOS:
    fuera = set(items[a]) - valores_validos(a)
    if fuera:
        raise ValueError(f"{a}: valores fuera del esquema {sorted(fuera)}")
print("Todos los valores están en el vocabulario del esquema.")

ejemplos = items.sample(12, random_state=3)[["slot", "name", *ATRIBUTOS]]
display(ejemplos)

Todos los valores están en el vocabulario del esquema.


,slot,name,tono,familia_color,patron,material,estilo,calce,epoca,largo
59185,bolso,valentino rockstud leather shoulder bag,desconocido,desconocido,liso,cuero,desconocido,no_aplica,contemporaneo,no_aplica
43577,vestido,true lace sweetheart skater dress,desconocido,desconocido,liso,encaje,desconocido,desconocido,contemporaneo,desconocido
4464,abajo,gucci butterfly pocket jeans,desconocido,desconocido,liso,denim,desconocido,desconocido,contemporaneo,desconocido
18060,accesorio,jimmy choo metal sunglasses,desconocido,desconocido,liso,desconocido,desconocido,no_aplica,contemporaneo,no_aplica
69317,exterior,topshop front boyfriend coat,desconocido,desconocido,liso,desconocido,desconocido,oversize,contemporaneo,desconocido
66832,vestido,boat neck flare sleeve pleated dress,desconocido,desconocido,liso,desconocido,desconocido,desconocido,contemporaneo,desconocido
14916,calzado,pre-owned aquazzura beverly lace-up sandals,desconocido,desconocido,liso,encaje,desconocido,no_aplica,contemporaneo,no_aplica
62383,joyeria,bvlgari 18kt diamond bangle,desconocido,desconocido,liso,desconocido,desconocido,no_aplica,contemporaneo,no_aplica
63582,arriba,seconds summer boxy tank design 5sos luke mich...,desconocido,desconocido,liso,desconocido,desconocido,oversize,contemporaneo,desconocido
56283,accesorio,ray-ban clubmaster classic sunglasses,desconocido,desconocido,liso,desconocido,desconocido,no_aplica,contemporaneo,no_aplica


## 2. Cobertura

Qué parte de las prendas tiene un valor leído del texto o de la categoría, cuántas quedan por supuesto (liso, contemporáneo) y cuántas quedan desconocidas. Calce y largo se miden solo sobre la ropa.

In [3]:
def fuente(origen):
    if origen in ("-", "sin nombre"):
        return "desconocido"
    if origen == "supuesto":
        return "supuesto"
    if origen == "regla":
        return "regla"
    if origen == "no aplica al slot":
        return "no aplica"
    return "categoria" if origen.startswith("categoria:") else "texto"

ropa = items["slot"].isin(SLOTS_ROPA)
filas = []
for a in ATRIBUTOS:
    base = items[ropa] if a in ("calce", "largo") else items
    f = base[f"origen_{a}"].map(fuente)
    if a == "familia_color":
        f = base["familia_color"].eq("desconocido").map({True: "desconocido", False: "regla"})
    cuenta = f.value_counts(normalize=True).mul(100).round(1)
    filas.append({"atributo": a, "base": "ropa" if a in ("calce", "largo") else "todas", **cuenta.to_dict()})
cobertura = pd.DataFrame(filas).set_index("atributo").fillna(0)
cobertura = cobertura[[c for c in ["base", "texto", "categoria", "regla", "supuesto", "desconocido"] if c in cobertura]]
display(cobertura)

print("Distribución de valores:")
for a in ATRIBUTOS:
    print(f"  {a}:", items[a].value_counts().head(8).to_dict())

,base,texto,categoria,regla,supuesto,desconocido
atributo,,,,,,
tono,todas,30.3,0.0,0.0,0.0,69.7
familia_color,todas,0.0,0.0,30.3,0.0,69.7
patron,todas,10.1,0.0,0.0,89.2,0.7
material,todas,32.2,0.1,0.0,0.0,67.7
estilo,todas,5.3,12.7,0.0,0.0,82.0
calce,ropa,14.1,3.1,0.0,0.0,82.8
epoca,todas,2.2,0.0,0.0,97.1,0.7
largo,ropa,21.3,6.2,0.0,0.0,72.5


Distribución de valores:
  tono: {'desconocido': 56720, 'negro': 6807, 'dorado': 3046, 'blanco': 2314, 'azul': 2143, 'rosa': 1555, 'rojo': 1475, 'plateado': 1324}
  familia_color: {'desconocido': 56720, 'neutro': 17604, 'fuerte': 6297, 'suave': 751}
  patron: {'liso': 72579, 'floral': 2295, 'estampado': 2187, 'rayas': 1340, 'cuadros': 854, 'animal': 720, 'desconocido': 542, 'grafico': 537}
  material: {'desconocido': 55090, 'cuero': 8236, 'denim': 5882, 'gamuza': 2548, 'encaje': 1956, 'lana': 1951, 'algodon': 1927, 'punto': 1583}
  estilo: {'desconocido': 66722, 'formal': 8345, 'athleisure': 3500, 'vintage': 1660, 'casual': 801, 'deportivo': 344}
  calce: {'no_aplica': 49043, 'desconocido': 26767, 'ajustado': 3532, 'oversize': 1739, 'recto': 291}
  epoca: {'contemporaneo': 79026, 'vintage_generico': 1665, 'desconocido': 542, '1990s': 73, '1980s': 39, '1970s': 16, '1950s': 11}
  largo: {'no_aplica': 49043, 'desconocido': 23435, 'corto': 5960, 'midi': 2041, 'largo': 893}


In [4]:
print("Tono conocido por slot (%):")
display(items.groupby("slot")["tono"].apply(lambda s: round(100 * s.ne("desconocido").mean(), 1)).to_frame("% con color"))

outfit_color = items.groupby("set_id")["tono"].apply(lambda s: s.ne("desconocido").mean())
print(f"Outfits donde se conoce el color de todas las prendas: {(outfit_color == 1).mean():.1%}")
print(f"Outfits donde se conoce el color de al menos la mitad: {(outfit_color >= 0.5).mean():.1%}")
print(f"Outfits sin ningún color conocido: {(outfit_color == 0).mean():.1%}")

Tono conocido por slot (%):


,% con color
slot,
abajo,26.8
accesorio,31.2
arriba,24.7
bolso,27.2
calzado,28.6
exterior,27.3
joyeria,45.4
vestido,24.0


Outfits donde se conoce el color de todas las prendas: 1.1%
Outfits donde se conoce el color de al menos la mitad: 24.4%
Outfits sin ningún color conocido: 20.0%


## 3. Salida

In [5]:
cols = ["item_key", "set_id", "index", "tid", "particion_oficial", "categoryid", "categoria", "slot", "genero",
        "name", "texto", *ATRIBUTOS, *[f"origen_{a}" for a in ATRIBUTOS]]
items[cols].to_csv(SALIDA, index=False, encoding="utf-8")
print("Guardado:", SALIDA.relative_to(ROOT), items[cols].shape)

Guardado: data\processed\hito3\polyvore_items_atributos.csv (81372, 27)
